# LLM Scoring Analysis: Axis A (DPM)

Analyzes the output of `scripts/score_axis_llm_ddr.py` -- loads a scoring run and produces the debiased mean-rating point estimate per post/concept (§6.3.2's aggregation rule).

**Order-effect / counterbalancing diagnostics live in `01_robustness_checks.ipynb`**, run once on a pilot subsample before committing to a full-corpus run -- this notebook assumes that check already happened and just aggregates whatever scoring run `SCORES_PATH` below points at (typically the full-corpus `--ordering-scheme single` pass, but it also works unchanged on a counterbalanced run).

**Reusable per axis**: this notebook is deliberately generic over the concept set. To analyze axis C (master frame) or axis D (justice arena) once their scoring runs complete, just change the lines in `## Config` below and rerun -- nothing else needs to change (§6.3.1's staged rollout: axis A first, C+D next, B last).

## Setup

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path("..").resolve()))

import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import seaborn as sns

from src import plot_style
from src.crisis_windows import CRISIS_WINDOWS, resolve_windows

plot_style.set_style()

## Config

Edit these to switch axis (`AXIS_NAME`/`SCORES_PATH`/`CONCEPT_NAMES`) and time-aggregation level (`AGG_LEVEL`, see below). `SCORES_PATH` must match the `--output` path used in the `score_axis_llm_ddr.py` run for that axis.

In [ ]:
# AXIS_NAME = "axis_a_dpm_de"
# SCORES_PATH = Path("../data/llm_scores/axis_a_dpm_de.csv")
# CONCEPT_NAMES = ["diagnostic", "prognostic", "motivational"]

AXIS_NAME = "axis_b_dpm_subcategories_de"
SCORES_PATH = Path("../data/llm_scores/axis_b_dpm_subcategories_de.csv")
CONCEPT_NAMES = ["blame_attribution", "problem_identification", "solutions", "tactics", "rationale", "call_to_action"]

# AXIS_NAME = "axis_c_masterframe_de"
# SCORES_PATH = Path("../data/llm_scores/axis_c_masterframe_de.csv")
# CONCEPT_NAMES = ["climate_change", "climate_justice"]


# AXIS_NAME = "axis_d_arena_de"
# SCORES_PATH = Path("../data/llm_scores/axis_d_arena_de.csv")
# CONCEPT_NAMES = ["oben_unten", "innen_aussen", "wir_sie", "heute_morgen"]


# Source corpus the scores were computed from -- used only to merge text back in for
# spot-checking (Load Scores below), since the scores file itself deliberately doesn't
# carry text (it's long-format, one row per doc PER ORDERING, so text would duplicate
# once per ordering if stored there -- see score_axis_llm_ddr.py's run_unit()).
SOURCE_PATH = Path("../data/processed/FFF_german.csv")
SOURCE_ID_COL = "id"
SOURCE_TEXT_COL = "text"


# Time-aggregation level for every plot below that bins posts by date (Frame Intensity Over
# Time, Relative Composition, the Cross-Axis plots). Pandas renamed several frequency aliases
# in 2.2 ("M" -> "ME" etc., the same deprecation that bit the AGG_FREQ = "M" bug earlier) --
# these are the current, non-deprecated ones. Change AGG_LEVEL and re-run the notebook from
# here down to compare granularities: weekly is the noisiest, yearly the smoothest (and, with
# only ~8 years of data, starts to be too few points to say much).
AGG_LEVELS = {
    "weekly": "W",
    "monthly": "ME",
    "quarterly": "QE",     # 3 months
    "semiannual": "6ME",   # 6 months
    "yearly": "YE",
}
AGG_LEVEL = "quarterly"
AGG_FREQ = AGG_LEVELS[AGG_LEVEL]

## Load Scores

Long format: one row per (post, ordering). A single-ordering run (`--ordering-scheme single`) will have exactly one `ordering_id` per post; a counterbalanced run will have several.

In [ ]:
scores = pd.read_csv(SCORES_PATH, dtype={"text_id": str})
n_orderings = scores["ordering_id"].nunique()
n_posts = scores["text_id"].nunique()
print(f"{len(scores)} rows -- {n_posts} posts x {n_orderings} ordering(s)")
print(f"orderings used: {sorted(scores['ordering'].unique())}")

n_missing = scores[CONCEPT_NAMES].isna().any(axis=1).sum()
print(f"rows with >=1 unparseable concept rating: {n_missing} ({n_missing / len(scores) * 100:.1f}%)")

# Merge text back in for spot-checking only -- NOT saved back to SCORES_PATH, this is
# just for display/inspection in this notebook (see Config cell for why it's not in
# the scores file itself).
source = pd.read_csv(SOURCE_PATH, dtype={SOURCE_ID_COL: str})[[SOURCE_ID_COL, SOURCE_TEXT_COL]]
scores_with_text = scores.merge(source, left_on="text_id", right_on=SOURCE_ID_COL, how="left")
scores_with_text.head()

## Debiased Point Estimates

Mean rating across orderings per post/concept -- this is the score to actually use downstream (not any single ordering's raw output), per §6.3.2's aggregation rule for intensity scores. For a `single`-ordering run this is just that ordering's rating, unchanged.

In [ ]:
final_scores = (
    scores.groupby("text_id")[CONCEPT_NAMES]
    .mean()
    .rename(columns={c: f"{c}_mean" for c in CONCEPT_NAMES})
)
if n_orderings > 1:
    sd_cols = scores.groupby("text_id")[CONCEPT_NAMES].std().rename(columns={c: f"{c}_sd" for c in CONCEPT_NAMES})
    final_scores = final_scores.join(sd_cols)

final_scores.head()

In [ ]:
plot_df = final_scores[[f"{c}_mean" for c in CONCEPT_NAMES]].melt(var_name="concept", value_name="mean_rating")
plot_df["concept"] = plot_df["concept"].str.replace("_mean", "", regex=False)

with plot_style.scaled_fonts(plot_style.FIGSIZE_WIDE):
    fig, ax = plt.subplots(figsize=plot_style.FIGSIZE_WIDE)
    sns.boxplot(data=plot_df, x="concept", y="mean_rating", ax=ax)
    ax.set_title(f"Distribution of Concept Ratings ({AXIS_NAME})")
    ax.set_xlabel("Concept")
    ax.set_ylabel("Mean rating across orderings")
    #plot_style.savefig(fig, f"FFF/{AXIS_NAME}_rating_distribution")
    plt.show()

## Frame Intensity Over Time (Core Plot)

Aggregates each concept's debiased rating by day or week and plots mean intensity as one line per concept, with shaded backgrounds for the crisis windows. This is the core plot for the project -- deliberately reusable across axes (nothing below hardcodes 3 concepts; change `AGG_FREQ` if needed and rerun for subcategories/master frame/inequality arena once those runs complete).

In [ ]:
dates = pd.read_csv(SOURCE_PATH, dtype={SOURCE_ID_COL: str})[[SOURCE_ID_COL, "creation_time"]]
dates["creation_time"] = pd.to_datetime(dates["creation_time"], utc=True)

scores_over_time = final_scores.reset_index().merge(
    dates, left_on="text_id", right_on=SOURCE_ID_COL, how="left"
)
n_unmatched = scores_over_time["creation_time"].isna().sum()
if n_unmatched:
    print(f"WARNING: {n_unmatched} scored posts have no matching date in {SOURCE_PATH} -- dropped from the time series")
scores_over_time = scores_over_time.dropna(subset=["creation_time"]).set_index("creation_time").sort_index()

intensity_over_time = (
    scores_over_time[[f"{c}_mean" for c in CONCEPT_NAMES]]
    .resample(AGG_FREQ)
    .mean()
    .rename(columns={f"{c}_mean": c for c in CONCEPT_NAMES})
)
intensity_over_time.head()

In [ ]:
plot_df = intensity_over_time.reset_index().melt(id_vars="creation_time", var_name="concept", value_name="mean_intensity")
resolved_windows = resolve_windows(CRISIS_WINDOWS, scores_over_time.index.max())

with plot_style.scaled_fonts(plot_style.FIGSIZE_HERO):
    fig, ax = plt.subplots(figsize=plot_style.FIGSIZE_HERO)

    window_colors = sns.color_palette("Pastel1", n_colors=len(resolved_windows))
    for (name, start, end, _description), color in zip(resolved_windows, window_colors):
        ax.axvspan(start, end, color=color, alpha=0.4, zorder=0)

    sns.lineplot(data=plot_df, x="creation_time", y="mean_intensity", hue="concept",
                 palette=plot_style.PALETTE_LINES, linewidth=1.8, ax=ax, zorder=3)

    ax.set_title(f"Frame Intensity Over Time ({AXIS_NAME}, {AGG_LEVEL})")
    ax.set_xlabel("Date")
    ax.set_ylabel("Mean intensity")

    # Two separate legends -- concept lines and crisis windows are different kinds of
    # information; stacking small separate legends reads more cleanly than one mixed one.
    concept_legend = ax.legend(loc="upper left", bbox_to_anchor=(1.02, 1), title="Frame")
    ax.add_artist(concept_legend)

    # Anchor the second legend to the first legend's ACTUAL rendered bottom edge (rather than
    # a fixed offset estimated from entry count) -- a fixed offset would need re-tuning any
    # time the legend font size changes, as scaled_fonts() makes it do per-figsize.
    fig.canvas.draw()
    concept_legend_bbox_axes = concept_legend.get_window_extent().transformed(ax.transAxes.inverted())
    window_legend_y = concept_legend_bbox_axes.y0 - 0.03

    window_handles = [Patch(facecolor=color, alpha=0.4, label=name) for (name, *_), color in zip(resolved_windows, window_colors)]
    ax.legend(handles=window_handles, loc="upper left", bbox_to_anchor=(1.02, window_legend_y),
              title="Crisis Window")

    #plot_style.savefig(fig, f"FFF/{AXIS_NAME}_intensity_over_time")
    plt.show()

### Relative Frame Composition Over Time

The plot above shows each frame's *absolute* intensity, but doesn't directly answer "did one frame intensify at the expense of another, or did they all move together?" -- two frames can both rise in absolute terms while one is still shrinking as a *share* of the total, or vice versa. This normalizes each time bin's concept intensities to sum to 1 and plots them as a 100%-stacked area -- a shrinking band for a concept means it's losing relative ground even if its raw line above isn't falling.

In [ ]:
row_sums = intensity_over_time[CONCEPT_NAMES].sum(axis=1)
share_over_time = intensity_over_time[CONCEPT_NAMES].div(row_sums, axis=0).dropna()  # empty time bins (no posts that period) drop out rather than plotting as a gap

with plot_style.scaled_fonts(plot_style.FIGSIZE_HERO):
    fig, ax = plt.subplots(figsize=plot_style.FIGSIZE_HERO)

    # No crisis-window shading here (unlike the plot above) -- the stacked fill covers the full
    # vertical span, so shading behind it was never actually visible.
    ax.stackplot(share_over_time.index, [share_over_time[c] for c in CONCEPT_NAMES], labels=CONCEPT_NAMES,
                 colors=plot_style.PALETTE_LINES[:len(CONCEPT_NAMES)], alpha=0.85, zorder=1)

    ax.set_ylim(0, 1)
    ax.set_title(f"Relative Frame Composition Over Time ({AXIS_NAME}, {AGG_LEVEL})")
    ax.set_xlabel("Date")
    ax.set_ylabel("Share of total intensity")

    ax.legend(loc="upper left", bbox_to_anchor=(1.02, 1), title="Frame")

    #plot_style.savefig(fig, f"FFF/{AXIS_NAME}_relative_composition")
    plt.show()

### Per-Window Summary

Concrete numbers to cite alongside the two plots above -- mean intensity per concept, per crisis window (not per time bin), so you can say e.g. "diagnostic averaged X in baseline vs. Y in Ukraine/energy" directly.

In [ ]:
window_summary_rows = []
for name, start, end, _description in resolved_windows:
    window_posts = scores_over_time.loc[start:end]
    row = {"window": name, "n_posts": len(window_posts)}
    for c in CONCEPT_NAMES:
        row[c] = window_posts[f"{c}_mean"].mean()
    window_summary_rows.append(row)

window_summary = pd.DataFrame(window_summary_rows).set_index("window")
window_summary

In [ ]:
plot_df = window_summary[CONCEPT_NAMES].reset_index().melt(id_vars="window", var_name="concept", value_name="mean_intensity")

# FIGSIZE_WIDE is narrower than FIGSIZE_HERO (which the notebook's font sizes are tuned
# for) -- scaled_fonts() keeps this plot's on-page text size consistent with the hero plots
# once both are placed at the same width in the report, even though it looks smaller here.
with plot_style.scaled_fonts(plot_style.FIGSIZE_WIDE):
    fig, ax = plt.subplots(figsize=plot_style.FIGSIZE_WIDE)
    sns.barplot(data=plot_df, x="window", y="mean_intensity", hue="concept", ax=ax)
    ax.set_title(f"Mean Intensity per Crisis Window ({AXIS_NAME})")
    ax.set_xlabel("Crisis window")
    ax.set_ylabel("Mean intensity")
    ax.tick_params(axis="x", rotation=20)
    ax.legend(title="Frame", loc="upper left", bbox_to_anchor=(1.02, 1))
    plot_style.savefig(fig, f"FFF/{AXIS_NAME}_window_summary")
    plt.show()

### More Plot Ideas (not built here)

Small multiples, a two-axis overlay, and a cross-axis correlation matrix are now built below ("Cross-Axis Analysis"). Remaining ideas:

- **Volatility/instability over time** -- rolling SD of each concept's weekly intensity (not SD across orderings, SD across *time*). A frame that's noisy week-to-week within a window reads differently than one that's stably high or stably low; the line plot above shows the mean trend but not how jumpy it is.
- **Window-to-window change, not just window levels** -- the bar chart above shows *levels* per window; a paired bar or slope chart of the *change* from the immediately preceding window per concept would more directly answer "what shifted when a crisis hit," rather than requiring the reader to eyeball adjacent bars.

## Cross-Axis Analysis

Everything above describes one axis at a time. These plots look *across* axes -- directly the empirical test for RQ3 (does the climate-justice shift move with DPM amplification, or not?) and RQ4 (does it move with arena extension?), rather than describing each axis's evolution in isolation. Needs all four axes' scoring runs to be complete.

In [ ]:
AXES_REGISTRY = {
    "axis_a_dpm_de": {
        "label": "DPM",
        "scores_path": Path("../data/llm_scores/axis_a_dpm_de.csv"),
        "concepts": ["diagnostic", "prognostic", "motivational"],
    },
    "axis_b_dpm_subcategories_de": {
        "label": "DPM Subcategories",
        "scores_path": Path("../data/llm_scores/axis_b_dpm_subcategories_de.csv"),
        "concepts": ["blame_attribution", "problem_identification", "solutions", "tactics", "rationale", "call_to_action"],
    },
    "axis_c_masterframe_de": {
        "label": "Master Frame",
        "scores_path": Path("../data/llm_scores/axis_c_masterframe_de.csv"),
        "concepts": ["climate_change", "climate_justice"],
    },
    "axis_d_arena_de": {
        "label": "Inequality Arena",
        "scores_path": Path("../data/llm_scores/axis_d_arena_de.csv"),
        "concepts": ["oben_unten", "innen_aussen", "wir_sie", "heute_morgen"],
    },
}


def load_axis_means(scores_path, concepts):
    """Debiased (mean-across-orderings) per-post score for one axis -- same rule as the
    single-axis Debiased Point Estimates cell above."""
    scores = pd.read_csv(scores_path, dtype={"text_id": str})
    return scores.groupby("text_id")[concepts].mean()


def load_axis_over_time(scores_path, concepts, freq):
    """Debiased per-post scores, merged with post dates (reuses `dates`, already loaded
    above), resampled to freq -- same pipeline as the single-axis Frame Intensity Over Time
    cell above, generalized to any axis.

    `freq` has no default deliberately -- always pass AGG_FREQ explicitly at the call site
    rather than defaulting to it. A default value is captured once, when this cell runs;
    changing AGG_LEVEL in Config and re-running only the plotting cells below (a very likely
    workflow when comparing aggregation levels) would silently keep using whatever AGG_FREQ
    was current when THIS cell last ran, not the new one.
    """
    means = load_axis_means(scores_path, concepts)
    merged = means.reset_index().merge(dates, left_on="text_id", right_on=SOURCE_ID_COL, how="left")
    merged = merged.dropna(subset=["creation_time"]).set_index("creation_time").sort_index()
    return merged[concepts].resample(freq).mean()

### All Axes, Stacked (Small Multiples)

One row per axis, all of that axis's concepts in color, sharing the x-axis so the panels line up vertically for comparison. Same crisis-window shading as the single-axis plot above, repeated in every panel deliberately -- vertical comparison is the point.

In [ ]:
small_multiples_figsize = (13, 4 * len(AXES_REGISTRY))

with plot_style.scaled_fonts(small_multiples_figsize) as scale:
    fig, axes = plt.subplots(len(AXES_REGISTRY), 1, figsize=small_multiples_figsize, sharex=True)

    for ax, axis_info in zip(axes, AXES_REGISTRY.values()):
        axis_over_time = load_axis_over_time(axis_info["scores_path"], axis_info["concepts"], AGG_FREQ)
        plot_df = axis_over_time.reset_index().melt(id_vars="creation_time", var_name="concept", value_name="mean_intensity")

        for (name, start, end, _description), color in zip(resolved_windows, window_colors):
            ax.axvspan(start, end, color=color, alpha=0.4, zorder=0)
        # Small, explicitly-scaled size -- window labels must fit inside their own often-narrow
        # date range, so they can't follow the general body-text scaling like everything else.
        plot_style.add_window_labels(ax, resolved_windows, window_colors, fontsize=3.5 * scale)

        sns.lineplot(data=plot_df, x="creation_time", y="mean_intensity", hue="concept",
                     palette=plot_style.PALETTE_LINES, linewidth=1.6, ax=ax, zorder=3)

        ax.set_title(axis_info["label"], loc="left", fontweight="bold")
        ax.set_ylabel("Mean intensity")
        ax.set_xlabel("")
        ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1), title=None)

    axes[-1].set_xlabel("Date")
    fig.suptitle(f"Frame Intensity Over Time -- All Axes ({AGG_LEVEL})", y=1.0)
    fig.tight_layout()
    plot_style.savefig(fig, "FFF/all_axes_intensity_small_multiples")
    plt.show()

### Two-Axis Overlay (placeholder)

Compares exactly two axes, each on its own y-axis (left/right). Both are the same 0-4 rating scale -- the second axis isn't there because the numeric ranges differ, it's there so each axis's lines stay visually distinct (solid vs. dashed, plus separate axis labels) rather than blending into one shared legend.

Currently set to Axis A vs. Axis B as a placeholder -- change `OVERLAY_AXIS_1`/`OVERLAY_AXIS_2` below to whichever pair the small-multiples/correlation-matrix views flag as worth a closer look (C vs. A for RQ3, C vs. D for RQ4), and re-run just this cell.

In [ ]:
OVERLAY_AXIS_1 = "axis_a_dpm_de"
OVERLAY_AXIS_2 = "axis_d_arena_de"

axis1_info = AXES_REGISTRY[OVERLAY_AXIS_1]
axis2_info = AXES_REGISTRY[OVERLAY_AXIS_2]
axis1_over_time = load_axis_over_time(axis1_info["scores_path"], axis1_info["concepts"], AGG_FREQ)
axis2_over_time = load_axis_over_time(axis2_info["scores_path"], axis2_info["concepts"], AGG_FREQ)

n_combined = len(axis1_info["concepts"]) + len(axis2_info["concepts"])
if n_combined > len(plot_style.PALETTE_LINES):
    print(f"WARNING: {n_combined} concepts combined > {len(plot_style.PALETTE_LINES)} colors in "
          f"PALETTE_LINES -- some lines will repeat a color, told apart only by solid-vs-dashed.")
axis1_colors = plot_style.PALETTE_LINES[:len(axis1_info["concepts"])]
axis2_colors = plot_style.PALETTE_LINES[len(axis1_info["concepts"]):n_combined]

with plot_style.scaled_fonts(plot_style.FIGSIZE_HERO) as scale:
    fig, ax1 = plt.subplots(figsize=plot_style.FIGSIZE_HERO)
    ax2 = ax1.twinx()

    for (name, start, end, _description), color in zip(resolved_windows, window_colors):
        ax1.axvspan(start, end, color=color, alpha=0.3, zorder=0)
    # Small, explicitly-scaled size -- window labels must fit inside their own often-narrow
    # date range, so they can't follow the general body-text scaling like everything else.
    plot_style.add_window_labels(ax1, resolved_windows, window_colors, fontsize=4.5 * scale)

    for concept, color in zip(axis1_info["concepts"], axis1_colors):
        ax1.plot(axis1_over_time.index, axis1_over_time[concept], color=color, linewidth=2.0, linestyle="-",
                 label=concept, zorder=3)
    for concept, color in zip(axis2_info["concepts"], axis2_colors):
        ax2.plot(axis2_over_time.index, axis2_over_time[concept], color=color, linewidth=2.0, linestyle="--",
                 label=concept, zorder=3)

    ax1.set_ylim(-0.2, 4)
    ax2.set_ylim(-0.2, 4)
    ax1.set_ylabel(f"{axis1_info['label']} (solid)")
    ax2.set_ylabel(f"{axis2_info['label']} (dashed)")
    ax1.set_xlabel("Date")
    ax1.set_title(f"{axis1_info['label']} vs. {axis2_info['label']} Over Time ({AGG_LEVEL})")

    # Two separate legends -- one per axis -- rather than one merged list, so the solid (axis 1)
    # and dashed (axis 2) concepts aren't read as one flat, undifferentiated set.
    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    legend1 = ax1.legend(lines1, labels1, loc="upper left", bbox_to_anchor=(1.1, 1), title=axis1_info["label"])
    ax1.add_artist(legend1)

    # Measure legend1's ACTUAL rendered extent rather than estimating its height from entry
    # count -- a fixed-per-row offset would need re-tuning any time the legend font size
    # changes (as scaled_fonts() makes it do per-figsize), so anchor legend2 to legend1's real
    # bottom edge instead, with a small gap, however tall legend1 actually turns out to be.
    fig.canvas.draw()
    legend1_bbox_axes = legend1.get_window_extent().transformed(ax1.transAxes.inverted())
    legend2_y = legend1_bbox_axes.y0 - 0.03
    ax1.legend(lines2, labels2, loc="upper left", bbox_to_anchor=(1.1, legend2_y), title=axis2_info["label"])

    plot_style.savefig(fig, f"FFF/overlay_{OVERLAY_AXIS_1}_vs_{OVERLAY_AXIS_2}")
    plt.show()

### Full Cross-Axis Correlation Matrix

Every concept against every other, across all four axes -- deliberately not collapsed to one line per axis. An axis-level summary could average away a real relationship that only shows up in one subframe (e.g. `motivational` moving with an arena while `diagnostic`/`prognostic` don't) -- same reasoning as `04_llm_agreement_validation.ipynb`'s DPM-vs-subcategory check, just extended to all four axes at once. Black lines mark the boundaries between axes (DPM / subcategories / master frame / arena), so the four blocks are visible without having to read every tick label.

In [ ]:
all_means = None
for axis_info in AXES_REGISTRY.values():
    means = load_axis_means(axis_info["scores_path"], axis_info["concepts"])
    all_means = means if all_means is None else all_means.join(means, how="inner")

print(f"{len(all_means)} posts scored on all {len(AXES_REGISTRY)} axes")

all_concepts = [c for axis_info in AXES_REGISTRY.values() for c in axis_info["concepts"]]
full_corr = all_means[all_concepts].corr()

corr_figsize = (12, 10)
with plot_style.scaled_fonts(corr_figsize) as scale:
    fig, ax = plt.subplots(figsize=corr_figsize)
    # annot_kws isn't driven by an rcParam scaled_fonts() sets -- scale it explicitly so the
    # cell annotations land on the same final on-page size as everything else in the figure.
    sns.heatmap(full_corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0, vmin=-1, vmax=1,
                annot_kws={"size": 6 * scale}, ax=ax)

    boundary = 0
    for axis_info in AXES_REGISTRY.values():
        boundary += len(axis_info["concepts"])
        ax.axhline(boundary, color="black", linewidth=1.5)
        ax.axvline(boundary, color="black", linewidth=1.5)

    ax.set_title("Correlation Matrix -- All Concepts, All Axes")
    plot_style.savefig(fig, "FFF/all_axes_correlation_matrix")
    plt.show()

## Key Takeaways: Quarterly Cross-Axis Exploration

Based on quarterly-aggregated exploration of the interactive dashboard and the static plots above -- each axis read individually and against every other axis, each read against its overall trend, the crisis-window periodization, and the key-event markers. Also folds in the earlier scratch notes taken directly under specific plots further up in this notebook (each merged bullet below is marked with the plot it came from). Raw working notes; ready to be turned into report prose.

### Cross-cutting patterns

Three groupings recur across the individual- and pairwise-axis notes below:

1. **Mobilization cluster** -- `motivational` (A), `call_to_action` and `tactics` (B). Consistently the highest-intensity concepts on their respective axes, track each other closely, and move anti-cyclically relative to the more diagnostic-oriented subcategories.
2. **Diagnostic/problem cluster** -- `diagnostic` (A), `heute_morgen` (D), `climate_change` (C). Track each other closely across most of the period. This is a genuine finding, not a coding-scheme artifact: `heute_morgen`'s definition (`concepts/axis_d_arena_de.json`) is actually framed around intergenerational responsibility and cost distribution ("wer trägt die Kosten der Transformation, und wer erbt morgen die Folgen") -- conceptually closer to `climate_justice`'s definition (responsibility, distributive justice) than to `climate_change`'s explicitly non-justice, technocratic framing. That `heute_morgen` nonetheless tracks `climate_change` more closely in practice is a real, somewhat counterintuitive pattern worth developing in the report.
3. **Status/justice cluster (from late 2024)** -- `oben_unten` and `wir_sie` (D) together with `climate_justice` (C), peaking together at the end of 2024 / beginning of 2026, while the diagnostic/problem cluster (including `climate_change`) dips at the same point. (See Caveat below on `oben_unten`/`climate_justice`.)

### Caveat: possible measurement overlap for `oben_unten` / `climate_justice`

Noted while reviewing the *Two-Axis Overlay* plot, and confirmed against the coding definitions: `oben_unten` is defined around socioeconomic distribution/redistribution (cue terms include "Reiche", "Konzerne", "Milliardäre", "Umverteilung" -- `concepts/axis_d_arena_de.json`), and `climate_justice` is defined around responsibility and distributive justice (cue terms include "Verantwortung", "Verteilungsgerechtigkeit" -- `concepts/axis_c_masterframe_de.json`). Their co-peaking at the end of 2024 / beginning of 2026 should be read with this conceptual overlap in mind -- plausibly part of the correlation reflects the coding scheme rather than two fully independent frames moving together.

### Axis A (DPM)

- **Overall**: `motivational` is consistently the most intense of the three, ranging roughly 2.0-3.5 across the whole period. `diagnostic` and `prognostic` sit lower (roughly 1.0-2.5) and track each other closely. This is intuitive -- much of FFF's Instagram communication functions as a mobilization call rather than problem-naming or solution-proposing.
- **By crisis window**: `diagnostic` and `prognostic` rose through baseline, stabilized through COVID-19 and the Ukraine War/Energy Crisis, dipped slightly during the cost-of-living crisis (through roughly mid-2024), then rose again. Within this, `prognostic` was slightly stronger during baseline, then `diagnostic` became slightly stronger afterward (see *Mean Intensity per Crisis Window* plot). `motivational` jumped from 2019 to 2020 and then stayed roughly stable for the rest of the period.
- **By key events**: elections coincide with high `motivational` framing and relative dips in `diagnostic`/`prognostic`. Global strikes also coincide with `motivational` peaks.

### Axis B (DPM Subcategories)

- **Overall**: `blame_attribution`, `solutions`, and `rationale` sit at a lower intensity than `problem_identification`, `tactics`, and `call_to_action`, which are consistently the strongest (also independently noted via the *Mean Intensity per Crisis Window* plot). `call_to_action` is the highest overall, followed by `tactics`. All subcategories except `tactics` and `call_to_action` dip around mid-2024. `call_to_action` was elevated for roughly nine consecutive months in both 2022 and 2024. `problem_identification` and `rationale` track each other closely; `call_to_action` rises and then holds roughly stable with a slight upward drift, while the other four rise, dip slightly around 2024, then rise again. Periods of high `call_to_action` tend to coincide with dips in `problem_identification`, `blame_attribution`, `solutions`, and `rationale`, and vice versa -- the subcategories don't all move together; rather, FFF's communication seems to alternate between a problem-identifying mode and a mobilizing mode.
- **By crisis window**: `call_to_action` dips at the end of 2019 (just ahead of the pandemic) and again around the German lockdowns (New Year's period). `solutions` peaks during the Ukraine War/Energy Crisis window.
- **By key events**: Bundestag elections, EU elections, and global strikes all coincide with high `call_to_action`.

### Axis C (Master Frame)

- **Overall**: `climate_justice` is consistently more intense than `climate_change`, and the gap widens slightly over time, especially during the Ukraine War/Energy Crisis. Both frames dip from the cost-of-living crisis through March 2024, then jump back up.
- **By crisis window**: both frames rise together during baseline, stabilize during COVID-19, then start to drift apart -- a divergence that continues through the Ukraine War/Energy Crisis window. Both drop in parallel from the cost-of-living crisis through March 2024.
- **By key events**: no clear pattern relative to individual key events.

### Axis D (Inequality Arena)

- **Overall**: `heute_morgen` (today-tomorrow) is the most intense arena for most of the period -- unsurprising, since climate framing is inherently future/intergenerational. Note that the whole axis sits at a much lower absolute intensity ceiling than the other three axes -- roughly 1.2-1.3 at most, versus 2-4 for DPM/subcategories/master frame (see *All Axes, Stacked (Small Multiples)* plot); keep this in mind when reading the two-axis overlay plots, where Axis D shares a 0-4 scale with its comparison axis and will always look comparatively flat. From the Ukraine War/Energy Crisis onward, and especially from the Gaza War onward, the other arenas intensify and at times exceed `heute_morgen`. `innen_aussen` shows a secondary peak around 2025 in addition to its Ukraine-era peak (see *All Axes, Stacked (Small Multiples)* plot). `wir_sie` (us-them) peaks at the beginning of 2024, coinciding with a dip in `climate_change`/`climate_justice` and a rise in `motivational` -- possibly reflecting tension between international and German FFF positioning around Gaza War solidarity debates (see *All Axes, Stacked (Small Multiples)* and *Two-Axis Overlay* plots); this reading is tentative. `oben_unten` (top-bottom) peaks at the end of 2024 and again at the beginning of 2026. `wir_sie` and `oben_unten` tend to move in parallel overall -- consistent with both being group/status-based framings of inequality, as opposed to `heute_morgen`'s temporal framing.
- **By crisis window**: `innen_aussen` (inside-outside) peaks during the Ukraine War/Energy Crisis window, plausibly reflecting solidarity/refugee-support framing (also independently noted via the *Mean Intensity per Crisis Window* plot).
- **By key events**: no clear pattern relative to individual key events.

### Axis A vs. B

- **Overall**: `motivational` (A) and `call_to_action` (B) are both high and track closely together. `tactics` (B) also moves closely with this pair and anti-cyclically relative to the other subcategories -- together, `motivational`, `call_to_action`, and `tactics` form the mobilization cluster (see above). Separately, `diagnostic` (A) tracks closely with `rationale` and `problem_identification` (B) (see *Two-Axis Overlay* plot).
- **By crisis window**: combined frame intensity rises through baseline, stabilizes through COVID-19, dips slightly across the cost-of-living crisis into the start of the Gaza War window, then rises again.
- **By key events**: elections coincide with high `motivational` framing.

### Axis A vs. C

- **Overall**: `climate_change` and `climate_justice` (C) broadly mirror the evolution of `diagnostic` and `prognostic` (A) -- independently corroborated via the *All Axes, Stacked (Small Multiples)* plot.
- **By crisis window / by key events**: not yet examined.

### Axis A vs. D

- **Overall**: `heute_morgen` (D) mirrors `diagnostic` (A) most closely, and more loosely also tracks `prognostic` and `motivational`. `wir_sie` shares peaks with `prognostic` specifically; `oben_unten` aligns more with `diagnostic`. Separately, the *Two-Axis Overlay* plot notes `innen_aussen` and `heute_morgen` both showing elevated activity in 2022 alongside `diagnostic` and `prognostic` -- a slightly different pairing than the crisis-window note below (which paired `innen_aussen` with `wir_sie`); both may hold simultaneously. That plot also notes `wir_sie`'s 2024 peak coinciding with a rise in `motivational`.
- **By crisis window**: `innen_aussen` and `wir_sie` both peak during the Ukraine War/Energy Crisis window. `wir_sie` rises further from 2024 onward, paralleling `oben_unten`'s peak at the end of 2024 / beginning of 2026.
- **By key events**: not yet examined.

### Axis B vs. C

- **Overall**: `tactics` (B) and `climate_justice` (C) peak together.
- **By crisis window / by key events**: not yet examined.

### Axis B vs. D

- **Overall**: in 2022, all four inequality arenas except `oben_unten` show peaks, coinciding with a sharp peak in `solutions` and smaller peaks in `call_to_action`, `problem_identification`, `blame_attribution`, and `rationale` (see *Two-Axis Overlay* plot).
- **By crisis window / by key events**: not yet examined.

### Axis C vs. D

- **Overall**: `heute_morgen` and `climate_change` track closely across the whole period -- notable given `heute_morgen`'s definition is actually framed around intergenerational responsibility and cost distribution, conceptually closer to `climate_justice` (see Cross-cutting patterns above); their tight empirical tracking is a genuine finding, not a definitional artifact. `climate_justice` evolves together with `oben_unten`, both peaking at the end of 2024 / beginning of 2026 (see Caveat above on `oben_unten`'s definition). `wir_sie` peaks at the same time as `oben_unten`, while the other concepts -- including `climate_change` and `climate_justice` -- dip at that point.
- **By crisis window / by key events**: not yet examined.


## Save Aggregated Scores

In [ ]:
output_path = Path(f"../data/processed/{AXIS_NAME}_aggregated.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)
final_scores.to_csv(output_path)
print(f"saved {len(final_scores)} posts to {output_path}")